In [1]:
from __future__ import annotations

In [2]:
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=9"

In [3]:
from dataclasses import dataclass
from functools import partial
from typing import Any, Self

import jax
import jax.numpy as jnp

from slub.handlers import (
    ScanSemantics,
    Uninitialized,
    default_cond_handler,
    default_jit_handler,
    default_primitive_handler,
    default_scan_handler,
    default_shard_map_handler,
    default_while_handler,
)
from slub.interpreter import (
    Dispatcher,
    HandlerResult,
    Interpreter,
    JaxprEqn,
    TracerValue,
    contains_subjaxprs,
    reinterpret,
)


@partial(
    jax.tree_util.register_dataclass,
    meta_fields=("metadata",),
    data_fields=("stack", "value"),
)
@dataclass(frozen=True)
class MyContext:
    metadata: tuple[Any, ...]
    stack: tuple[Self, ...]
    value: tuple[TracerValue, ...]

    def add_meta(self, key: str) -> "MyContext":
        return MyContext(self.metadata + (key,), self.stack, self.value)

    def add_value(self, value: TracerValue) -> "MyContext":
        return MyContext(self.metadata, self.stack, self.value)

    def push(self) -> "MyContext":
        return MyContext((), self.stack + (self,), ())

    def pop(self) -> "MyContext":
        if len(self.stack) == 0:
            raise ValueError("Cannot pop root context")
        return MyContext(
            self.stack[-1].metadata + (self.metadata,),
            self.stack[-1].stack,
            self.stack[-1].value + self.value,
        )


def my_sin_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    ctx = ctx.add_meta("sin")
    ctx = ctx.add_value(jnp.zeros(8))
    return default_primitive_handler(_, ctx, eqn, invals)


def my_cos_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    ctx = ctx.add_meta("cos")
    ctx = ctx.add_value(jnp.zeros(7))
    return default_primitive_handler(_, ctx, eqn, invals)


def my_jit_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    # print(eqn.params["jaxpr"])
    ctx = ctx.add_meta("jit")
    ctx = ctx.add_value(jnp.ones(1, dtype=jnp.int32))
    return default_jit_handler(_, ctx, eqn, invals)


def my_scan_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    def initializer(old: MyContext, new: MyContext):
        tree = MyContext(
            new.metadata,
            old.stack + (old,),
            new.value,
        )
        leaves, tree_def = jax.tree.flatten(tree)
        assert tree_def == jax.tree.structure(new), (tree_def, jax.tree.structure(new))
        leaves = [
            jnp.zeros_like(leaf) if isinstance(leaf, Uninitialized) else leaf
            for leaf in leaves
        ]

        return jax.tree.unflatten(tree_def, leaves)

    def updater(_: MyContext, new: MyContext):
        return new

    ctx = ctx.add_meta("scan")
    ctx = ctx.add_value(jnp.zeros(1, dtype=jnp.float32))
    return default_scan_handler(
        _,
        ctx,
        eqn,
        invals,
        threading=ScanSemantics.CARRY,
        initializer=initializer,
        updater=updater,
    )


def my_cond_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    ctx = ctx.add_meta("cond")
    return default_cond_handler(_, ctx, eqn, invals)


def my_while_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    ctx = ctx.add_meta("while")

    def initializer(old: MyContext, new: MyContext):
        tree = MyContext(
            new.metadata,
            old.stack + (old,),
            new.value,
        )
        leaves, tree_def = jax.tree.flatten(tree)
        assert tree_def == jax.tree.structure(new), (tree_def, jax.tree.structure(new))
        leaves = [
            jnp.zeros_like(leaf) if isinstance(leaf, Uninitialized) else leaf
            for leaf in leaves
        ]

        return jax.tree.unflatten(tree_def, leaves)

    def updater(old: MyContext, new: MyContext):
        return new

    return default_while_handler(
        _, ctx, eqn, invals, initializer=initializer, updater=updater
    )


def my_shard_map_handler(
    _: Interpreter[MyContext],
    ctx: MyContext,
    eqn: JaxprEqn,
    invals: list[TracerValue],
) -> HandlerResult[MyContext]:
    ctx = ctx.add_meta("shard_map")

    def declare_ctx_out_specs(ctx: MyContext):
        return jax.tree.map(lambda _: jax.P(), ctx)

    return default_shard_map_handler(
        _, ctx, eqn, invals, declare_ctx_out_specs=declare_ctx_out_specs
    )


dispatcher = Dispatcher(
    {
        jax.lax.sin_p: my_sin_handler,
        jax.lax.cos_p: my_cos_handler,
        "jit": my_jit_handler,
        "scan": my_scan_handler,
        "cond": my_cond_handler,
        "while": my_while_handler,
        "shard_map": my_shard_map_handler,
    }
).register_custom_matching_rule(contains_subjaxprs)

interpreter = Interpreter(dispatcher=dispatcher)

In [4]:
import functools
from collections.abc import Callable


def test(
    x: jax.Array, y: jax.Array, z1: jax.Array, z2: jax.Array
) -> tuple[tuple[jax.Array, jax.Array], jax.Array]:
    @jax.jit
    def pass_through(x):
        return x

    @jax.jit
    def reused(x, y):
        return jnp.sin(x) + pass_through(y)

    u = reused(x, y)
    v = reused(x, y)

    def scan(carry, xs):
        @jax.jit
        def inner(carry, xs):
            const = z1.astype(jnp.float32).sum() + z2.astype(jnp.float32).sum()
            x, y = carry
            x = jnp.sin(x) + xs.sum()[None]
            y = jnp.sin(y) + y.sum() + const
            return (x, y), x.sum() + y.sum() + const

        return inner(carry, xs)

    return jax.lax.scan(scan, (u, v), jnp.ones((11, 1)))

In [5]:
MESH = jax.sharding.Mesh(jax.devices(), ("batch",))
BATCH_P = jax.P("batch")
BATCH_SHARDING = jax.NamedSharding(MESH, BATCH_P)


def shard_map[C: Callable](fun: C, /, out_specs, in_specs, **shard_kwargs) -> C:
    def inner_shard(fun: C) -> C:
        sharded_f = jax.shard_map(
            fun, out_specs=out_specs, in_specs=in_specs, **shard_kwargs
        )  # type: ignore

        @functools.wraps(fun)
        def wrapper(*args, **kwargs):
            return sharded_f(*args, **kwargs)

        return wrapper  # type: ignore

    if fun is None:
        return inner_shard
    return inner_shard(fun)


def vmap[C: Callable](f: C) -> C:
    sharded_f = jax.jit(
        shard_map(
            jax.vmap((f)),
            out_specs=BATCH_P,
            in_specs=BATCH_P,
            mesh=MESH,
        ),
        in_shardings=BATCH_SHARDING,
        out_shardings=BATCH_SHARDING,
    )

    @jax.jit
    def wrapped(*args, **kwargs):
        ctx_f = partial(reinterpret, interpreter=interpreter)(sharded_f)
        ctx = MyContext((), (), ())
        result, ctx = ctx_f(ctx, *args, **kwargs)
        return result, ctx

    return wrapped


out, ctx = vmap(test)(
    jnp.ones((18, 5)),
    jnp.ones((18, 5)),
    jnp.array(1.0) * jnp.ones(18),
    jnp.array(2.0) * jnp.ones(18),
)
print("Result:", out)

Result: ((Array([[1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 1.9345641, 1.9345641, 1.9345641],
       [1.9345641, 1.9345641, 